# 12A · Diversification & the Efficient Frontier
### Financial Analytics — Module 12 · Lab 4

The course's final summit, and the payoff of a chain of promises: Module 3's √252 and correlation, Module 4's heatmap ("the diversification map"), Module 7's optimiser anatomy — all converge on one of the few genuinely Nobel-winning ideas simple enough to build in an afternoon:

> **A portfolio's risk depends not just on what you hold, but on how the holdings move TOGETHER.**

1. Two assets: watch correlation bend the risk-return line into a curve
2. Twenty stocks: the cloud of every possible portfolio
3. The **efficient frontier**: the cloud's north-west edge, found by optimisation
4. The two celebrity portfolios: minimum-variance and maximum-Sharpe

> 🛡️ **Bias check:** inputs are estimated from 4 years of a SURVIVOR universe (today's large caps — stated, optimistic). TATAMOTORS split back-adjusted before anything else (11B's lesson, now standard practice). Regime: one bull-tilted window; expected returns estimated from it are the shakiest input we will use — 12B is entirely about that shakiness. Look-ahead: none within this notebook; 12B tests what happens when there effectively was.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(12)

import os
BASE = "data/" if os.path.exists("data") else "https://raw.githubusercontent.com/vivekhashtag/financial-analytics-course/main/data/"
uni = pd.read_csv(BASE + "nse_stock_universe.csv", parse_dates=["date"])
px = uni.pivot(index="date", columns="ticker", values="close").sort_index()
px.loc[:"2024-09-01", "TATAMOTORS.NS"] = px.loc[:"2024-09-01", "TATAMOTORS.NS"] / 5   # 11B's fix: reflex now

rets = px.pct_change().dropna()
mu  = rets.mean()*252                # annualised expected returns (the fragile input!)
cov = rets.cov()*252                 # annualised covariance matrix (the sturdier one)
print(f"{rets.shape[1]} stocks, {rets.shape[0]} days | avg pairwise corr: {rets.corr().values[np.triu_indices(20,1)].mean():.2f}")

---
## 1. Two assets, one dial: correlation

Take two stocks. Mix them in every proportion and plot the portfolio's risk and return — then turn the correlation dial and watch the shape change:

In [ ]:
def port_stats(w, mu_v, cov_m):
    ret = w @ mu_v
    vol = np.sqrt(w @ cov_m @ w)
    return ret, vol

# Two real stocks' stats, but we'll IMPOSE different correlations to see the pure effect
a, b = "HDFCBANK.NS", "INFY.NS"
r1, r2 = mu[a], mu[b]
v1, v2 = rets[a].std()*np.sqrt(252), rets[b].std()*np.sqrt(252)

fig, ax = plt.subplots(figsize=(8.5, 4.5))
ws = np.linspace(0, 1, 101)
for corr, clr in [(1.0, "#94A3B8"), (0.5, "#2563EB"), (0.0, "#0D9488"), (-0.5, "#DB2777")]:
    cov2 = np.array([[v1**2, corr*v1*v2], [corr*v1*v2, v2**2]])
    pts = [port_stats(np.array([w, 1-w]), np.array([r1, r2]), cov2) for w in ws]
    ax.plot([p[1] for p in pts], [p[0] for p in pts], color=clr, lw=2, label=f"corr = {corr:+.1f}")
ax.set_xlabel("volatility (risk)"); ax.set_ylabel("expected return")
ax.set_title("Two assets, four correlations: lower correlation BENDS the line left - risk vanishes without\nsacrificing return. This bend is the only free lunch in finance.", loc="left", fontweight="bold", fontsize=10)
ax.legend(); plt.tight_layout(); plt.show()

**Read the bend.** At correlation +1.0, mixing buys nothing — risk is a straight average. As correlation falls, the curve bows *leftward*: portfolios exist with **less risk than either ingredient**, at no cost in return. The wobbles partially cancel. That leftward bend — bought purely by combining things that don't move together — is why diversification is called the only free lunch in finance, and the entire lab is the industrial version of this one picture.

---
## 2. Twenty stocks: the cloud of all portfolios

In [ ]:
# 8,000 random long-only portfolios: what does the space of choices look like?
N = 8_000
W = rng.dirichlet(np.ones(20), N)          # random weights, each row sums to 1, all >= 0
cloud_ret = W @ mu.values
cloud_vol = np.sqrt(np.einsum("ij,jk,ik->i", W, cov.values, W))
sharpe = cloud_ret / cloud_vol             # (risk-free ~0 kept for simplicity; stated)

fig, ax = plt.subplots(figsize=(9, 5))
sc = ax.scatter(cloud_vol, cloud_ret, c=sharpe, s=6, cmap="viridis", alpha=0.6)
plt.colorbar(sc, label="return / risk (Sharpe)")
# single stocks for reference
ax.scatter(np.sqrt(np.diag(cov)), mu.values, marker="x", color="#DC2626", s=40, label="single stocks")
ax.set_xlabel("volatility"); ax.set_ylabel("expected return")
ax.set_title("8,000 random portfolios (the cloud) vs the 20 single stocks (red x)", loc="left", fontweight="bold")
ax.legend(); plt.tight_layout(); plt.show()

print(f"Average single-stock vol: {np.sqrt(np.diag(cov)).mean():.1%}")
print(f"Average random-PORTFOLIO vol: {cloud_vol.mean():.1%}   <- diversification working before any optimisation")

Two facts hiding in the cloud: **every red × (single stock) sits deep inside or right of the cloud** — almost any random mix beats almost any single holding on risk; and the cloud has a hard **north-west edge** — for each level of risk, a best achievable return. Nobody can stand north-west of that edge. It has a name:

## 3. The efficient frontier — found by Module 7's machinery

Same anatomy as the treasury problem: decision variables (weights), objective (minimise variance), constraints (weights sum to 1, no shorting). New solver — `minimize` instead of `linprog` — because variance is quadratic, not linear. Everything else you already know.

In [ ]:
n = 20
cons_base = [{"type": "eq", "fun": lambda w: w.sum() - 1}]
bounds = [(0, 1)]*n
w0 = np.ones(n)/n

def vol_of(w): return np.sqrt(w @ cov.values @ w)

# Trace the frontier: for each target return, find the LOWEST-risk portfolio achieving it
targets = np.linspace(mu.values.min()+0.01, mu.values.max()-0.01, 30)
frontier = []
for t in targets:
    cons = cons_base + [{"type": "eq", "fun": lambda w, t=t: w @ mu.values - t}]
    res = minimize(vol_of, w0, bounds=bounds, constraints=cons, method="SLSQP")
    if res.success: frontier.append((res.fun, t))
f_vol, f_ret = zip(*frontier)

# The two celebrity portfolios
res_minvar = minimize(vol_of, w0, bounds=bounds, constraints=cons_base, method="SLSQP")
res_maxsh  = minimize(lambda w: -(w @ mu.values)/vol_of(w), w0, bounds=bounds, constraints=cons_base, method="SLSQP")
w_minvar, w_maxsh = res_minvar.x, res_maxsh.x

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(cloud_vol, cloud_ret, c=sharpe, s=5, cmap="viridis", alpha=0.35)
ax.plot(f_vol, f_ret, color="#DC2626", lw=2.5, label="efficient frontier")
ax.scatter(*port_stats(w_minvar, mu.values, cov.values)[::-1], marker="*", s=300, color="#2563EB",
           edgecolor="white", zorder=5, label="min-variance")
ax.scatter(*port_stats(w_maxsh, mu.values, cov.values)[::-1], marker="*", s=300, color="#EA580C",
           edgecolor="white", zorder=5, label="max-Sharpe")
ax.set_xlabel("volatility"); ax.set_ylabel("expected return")
ax.set_title("The frontier: the cloud's unreachable-beyond edge, traced by 30 optimisations",
             loc="left", fontweight="bold")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# What do the celebrity portfolios actually HOLD? (Always look at the weights - Module 7's binding-constraint habit)
weights = pd.DataFrame({"min_variance": w_minvar, "max_sharpe": w_maxsh},
                       index=[t.replace(".NS","") for t in px.columns])
big = weights[(weights > 0.01).any(axis=1)].sort_values("max_sharpe", ascending=False)
print((big*100).round(1).to_string())
print(f"\nMin-var: {(w_minvar>0.01).sum()} meaningful holdings, tilted to low-vol defensives.")
print(f"Max-Sharpe: {(w_maxsh>0.01).sum()} holdings - CONCENTRATED in whatever had the best estimated returns.")
print("That concentration should make you nervous. Hold the nerve until 12B.")

*(The course app renders this as an interactive 3D frontier — risk × return × Sharpe, rotatable, with your cursor picking portfolios and showing their weights live. The static twin above carries the same truth.)*

### ✏️ Exercises
1. **The free-lunch counter:** compute equal-weight portfolio vol using only the 5 IT+Financials stocks, then using 5 stocks from 5 different sectors. Same count, different correlation structure — how much vol does sector spread buy? (4B's heatmap, cashed in.)
2. **Constraint humility:** re-find max-Sharpe with a 10% cap per stock (`bounds=[(0, 0.10)]*n`). How much Sharpe is "lost"? How many holdings now? Which portfolio would you rather explain to a client after a bad quarter?
3. **The 3D twin:** make a 3D scatter (`ax = fig.add_subplot(projection='3d')`) of the cloud with axes vol/return/Sharpe. Rotate mentally: why is max-Sharpe the "peak of the ridge" in this view?

---
*AI disclosure: ______*

In [ ]:
# workspace
